# Phase 3 dataset analysis

Integrity and tokenization checks for the pinned Phase 2 release used by both experiments.

All numbers come from the exported run artifacts in `results/`. The current cohort is the provisional `gemini-v2` `span_checked` release (322 test question groups, human review pending), so results must be labelled provisional.

In [ ]:
import csv, json
from pathlib import Path
import matplotlib.pyplot as plt

RESULTS = Path.cwd().parent / "results" if Path.cwd().name == "notebooks" else Path("phase_03_multilingual_finetuning/results")
LANGS = ["en", "hi", "mr", "bn", "ta"]

def load_json(name):
    return json.loads((RESULTS / name).read_text(encoding="utf-8"))

def load_csv(name):
    with (RESULTS / name).open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))

def show(rows, fields=None, digits=2):
    """Print a plain-text table without extra dependencies."""
    fields = fields or list(rows[0])
    def fmt(value):
        try:
            return f"{float(value):.{digits}f}" if "." in str(value) else str(value)
        except ValueError:
            return str(value)
    cells = [[fmt(row.get(f, "")) for f in fields] for row in rows]
    widths = [max(len(f), *(len(c[i]) for c in cells)) for i, f in enumerate(fields)]
    print("  ".join(f.ljust(w) for f, w in zip(fields, widths)))
    for c in cells:
        print("  ".join(v.ljust(w) for v, w in zip(c, widths)))

def require(*names):
    missing = [n for n in names if not (RESULTS / n).exists()]
    if missing:
        raise FileNotFoundError("Run scripts/export_results.py first. Missing: " + ", ".join(missing))

## Dataset identity and integrity

In [ ]:
integrity = load_json("dataset_integrity_report.json")
print(json.dumps(integrity["dataset"], indent=2))
show([{"split": s, **integrity["counts_by_language"][s]} for s in ("train", "validation", "test")])
print("No cross-split question IDs:", integrity["no_cross_split_question_ids"])
print("All stored spans exact:", integrity["all_stored_spans_exact"])
print("Test rows whose exact passage also appears in train:", integrity["test_rows_with_exact_train_context"])

## Tokenization (XLM-R, 384 tokens, stride 128)

In [ ]:
tokens = load_json("tokenization_statistics.json")["by_split_and_language"]
rows = []
for split, by_lang in tokens.items():
    for lang in LANGS:
        s = by_lang[lang]
        rows.append({"split": split, "language": lang, "examples": s["examples"], "windows": s["windows"],
                     "mean_context_tokens": s["mean_context_tokens"], "max_context_tokens": s["max_context_tokens"],
                     "gold_over_decode_cap": len(s["gold_exceeds_decode_limit"])})
show(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
test = tokens["test"]
ax.bar([l.upper() for l in LANGS], [test[l]["mean_context_tokens"] for l in LANGS], color="#416e9a")
ax.set_ylabel("Mean context tokens (test)")
ax.set_title("Subword length of the same passages by language")
plt.show()